## Iniciando o Spark e importando bibliotecas

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("AnaliseDadosPos") \
    .master("spark://spark-master:7077") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/13 14:29:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
pos_cash = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/POS_CASH_balance.csv")
pos_cash.createOrReplaceTempView("pos_cash")

pos_cash.show(5, truncate=False)

+----------+----------+--------------+--------------+---------------------+--------------------+------+----------+
|SK_ID_PREV|SK_ID_CURR|MONTHS_BALANCE|CNT_INSTALMENT|CNT_INSTALMENT_FUTURE|NAME_CONTRACT_STATUS|SK_DPD|SK_DPD_DEF|
+----------+----------+--------------+--------------+---------------------+--------------------+------+----------+
|1803195   |182943    |-31           |48.0          |45.0                 |Active              |0     |0         |
|1715348   |367990    |-33           |36.0          |35.0                 |Active              |0     |0         |
|1784872   |397406    |-32           |12.0          |9.0                  |Active              |0     |0         |
|1903291   |269225    |-35           |48.0          |42.0                 |Active              |0     |0         |
|2341044   |334279    |-35           |36.0          |35.0                 |Active              |0     |0         |
+----------+----------+--------------+--------------+---------------------+-----

In [4]:
pos_cash.count()

10001358

In [14]:
pos_cash = spark.sql("""
                                Select
                                    SK_ID_PREV as PK_AGG_POS,
                                    SK_ID_CURR,
                                    CNT_INSTALMENT as FVL_CNT_INSTALMENT_POS,
                                    CNT_INSTALMENT_FUTURE as FVL_CNT_INSTALMENT_FUTURE_POS,
                                    NAME_CONTRACT_STATUS as FC_NAME_CONTRACT_STATUS_POS,
                                    SK_DPD as FVL_SK_DPD_POS,
                                    SK_DPD_DEF as FVL_SK_DPD_DEF_POS,
                                    cast(add_months('2023-12-01',MONTHS_BALANCE) as timestamp) as PK_DATREF_POS,
                                    substr(translate(cast(add_months('2023-12-01',MONTHS_BALANCE) as string),'-',''),1,6) as PK_DATPART_POS,
                                    MONTHS_BALANCE
                                from 
                                    pos_cash
                            """)
# Retirando valores nulos
pos_cash = pos_cash.where(col("MONTHS_BALANCE").isNotNull())

# Filtrando somente histórico necessário (15 meses)
stage01 = pos_cash.where(col("MONTHS_BALANCE") >= -200)
stage01 = stage01.drop("MONTHS_BALANCE")

stage01.createOrReplaceTempView("stage01")
stage01.count()

10001358

In [15]:
stage01.show(5, truncate=False)

+----------+----------+----------------------+-----------------------------+---------------------------+--------------+------------------+-------------------+--------------+
|PK_AGG_POS|SK_ID_CURR|FVL_CNT_INSTALMENT_POS|FVL_CNT_INSTALMENT_FUTURE_POS|FC_NAME_CONTRACT_STATUS_POS|FVL_SK_DPD_POS|FVL_SK_DPD_DEF_POS|PK_DATREF_POS      |PK_DATPART_POS|
+----------+----------+----------------------+-----------------------------+---------------------------+--------------+------------------+-------------------+--------------+
|1803195   |182943    |48.0                  |45.0                         |Active                     |0             |0                 |2021-05-01 00:00:00|202105        |
|1715348   |367990    |36.0                  |35.0                         |Active                     |0             |0                 |2021-03-01 00:00:00|202103        |
|1784872   |397406    |12.0                  |9.0                          |Active                     |0             |0          

In [16]:
spark.sql("""
            select
                PK_DATREF_POS,
                PK_DATPART_POS,
                count(*) as Volume
            from stage01
            group by 1,2
            order by  1 desc
""").show(200)

+-------------------+--------------+------+
|      PK_DATREF_POS|PK_DATPART_POS|Volume|
+-------------------+--------------+------+
|2023-11-01 00:00:00|        202311| 94908|
|2023-10-01 00:00:00|        202310|169529|
|2023-09-01 00:00:00|        202309|183589|
|2023-08-01 00:00:00|        202308|193147|
|2023-07-01 00:00:00|        202307|200726|
|2023-06-01 00:00:00|        202306|206849|
|2023-05-01 00:00:00|        202305|210229|
|2023-04-01 00:00:00|        202304|214149|
|2023-03-01 00:00:00|        202303|215558|
|2023-02-01 00:00:00|        202302|216441|
|2023-01-01 00:00:00|        202301|216023|
|2022-12-01 00:00:00|        202212|214716|
|2022-11-01 00:00:00|        202211|210950|
|2022-10-01 00:00:00|        202210|208352|
|2022-09-01 00:00:00|        202209|204935|
|2022-08-01 00:00:00|        202208|200432|
|2022-07-01 00:00:00|        202207|195713|
|2022-06-01 00:00:00|        202206|190385|
|2022-05-01 00:00:00|        202205|184302|
|2022-04-01 00:00:00|        202

In [17]:
spark.sql("""
            select
                PK_DATREF_POS,
                PK_DATPART_POS,
                count(*) as Volume
            from stage01
            group by 1,2
            order by  1 desc
""").count()

96

#### Salvando tabela particionada (Parquet)

In [19]:
nm_path = '/data/processed/pos_cash/'
stage01.write.partitionBy('PK_DATPART_POS').parquet(nm_path, mode='overwrite')

In [20]:
spark.stop()